# Tuning

Find the optimal `n_clusters` and `n_segments` for your data.

Three functions cover different use cases:

| Function | What it does |
|---|---|
| `find_best_combination` | Full grid search — returns best RMSE with complete history |
| `find_pareto_front` | Same grid, filtered to Pareto-optimal points |
| `find_optimal_combination` | Boundary curve for a target data reduction — fastest |

Unlike tsam's built-in tuning, all three evaluate each candidate across **all slices**.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook_connected"

da = sample_energy_data(n_days=30)
print(f"Shape: {dict(da.sizes)}")

## Grid search &amp; RMSE heatmap

In [ ]:
grid = tsam_xarray.find_best_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    max_timesteps=48,
    show_progress=False,
)
print(f"Tested {len(grid.history)} combinations")
print(f"Best: {grid.n_clusters}c x {grid.n_segments}s (RMSE={grid.rmse:.4f})")

grid.summary_matrix["rmse"].plotly.imshow(
    x="n_segments",
    y="n_clusters",
    title="RMSE by (n_clusters, n_segments)",
)

## Pareto front

`find_pareto_front` runs the same grid search but filters to non-dominated
configurations — those where no other combo has both lower RMSE *and* fewer timesteps.

In [ ]:
pareto = tsam_xarray.find_pareto_front(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    max_timesteps=48,
    show_progress=False,
    save_all_results=True,
)
print(f"Pareto-optimal: {len(pareto.history)} of {len(grid.history)} tested")
pareto.summary

In [ ]:
pareto.plot()

Compare cluster representatives at three points along the frontier:
the smallest, a mid-range, and the best configuration.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pick three points: smallest, mid, best
indices = [0, len(pareto) // 2, len(pareto) - 1]
fig = make_subplots(
    rows=1,
    cols=len(indices),
    subplot_titles=[
        f"{pareto.history[i]['n_clusters']}c x "
        f"{pareto.history[i]['n_segments']}s "
        f"(RMSE={pareto.history[i]['rmse']:.3f})"
        for i in indices
    ],
)
for col, idx in enumerate(indices, 1):
    cr = pareto[idx].cluster_representatives.sel(
        scenario="low", variable="solar", region="north"
    )
    for cl in cr.coords["cluster"].values:
        vals = cr.sel(cluster=cl)
        fig.add_trace(
            go.Scatter(
                x=vals.coords["timestep"].values,
                y=vals.values,
                mode="lines",
                line_shape="hv",
                name=f"cluster {cl}",
                showlegend=(col == 1),
            ),
            row=1,
            col=col,
        )
fig.update_layout(
    title="Pareto front: solar (north, low) at three complexity levels",
    height=350,
)
fig.show()

## Target data reduction

`find_optimal_combination` tests only the boundary combos for a target reduction — faster.

In [ ]:
result_opt = tsam_xarray.find_optimal_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    data_reduction=0.05,
    show_progress=False,
)
print(f"Best for 5% reduction: {result_opt.n_clusters}c x {result_opt.n_segments}s")
print(f"RMSE: {result_opt.rmse:.4f}")
result_opt.summary